In [1]:
import pandas as pd
import pubchempy as pcp
from tqdm import tqdm
import os
import random
from itertools import product

In [2]:
snap_df = pd.read_excel('../SNAP/final_data/SNAP.xlsx')
drh_df = pd.read_excel('../DRH/final_data/DRH.xlsx')
ttd_df = pd.read_excel('../TTD/final_data/TTD.xlsx')

print(snap_df.columns)
print(drh_df.columns)
print(ttd_df.columns)

Index(['DRUGID', 'UNIPROTID', 'DRUGNAME', 'SMILES', 'INCHIKEY', 'GENENAME',
       'SEQUENCE'],
      dtype='object')
Index(['DRUGNAME', 'GENENAME', 'UNIPROTID', 'SEQUENCE', 'CID', 'INCHIKEY',
       'SMILES', 'DRUGID'],
      dtype='object')
Index(['TARGETID', 'FORMERID', 'ENTRYNAME', 'TARGNAME', 'GENENAME', 'TARGTYPE',
       'BIOCLASS', 'ECNUMBER', 'DRUGID', 'DRUGNAME', 'CLINICAL_STATUS',
       'PDBSTRUC', 'UNIPROTID', 'SEQUENCE', 'TRADNAME', 'DRUGCOMP', 'THERCLAS',
       'DRUGTYPE', 'DRUGINCH', 'INCHIKEY', 'SMILES', 'HIGHSTAT', 'COMPCLAS'],
      dtype='object')


In [3]:
cols = ['INCHIKEY', 'UNIPROTID', 'DRUGNAME', 'SMILES', 'GENENAME', 'SEQUENCE']
sub_snap_df = snap_df[cols]
sub_snap_df['SOURCE'] = 'SNAP'

sub_drh_df = drh_df[cols]
sub_drh_df['SOURCE'] = 'DRH'

sub_ttd_df = ttd_df[cols]
sub_ttd_df['SOURCE'] = 'TTD'

/tmp/ipykernel_394025/3813629170.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_drh_df['SOURCE'] = 'DRH'
/tmp/ipykernel_394025/3813629170.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_ttd_df['SOURCE'] = 'TTD'


In [4]:
# 拼接三个数据集
all_pair_df = pd.concat([sub_snap_df, sub_drh_df, sub_ttd_df], axis=0, ignore_index=True)
print('Before deduplication:', len(all_pair_df))

# 根据药物–蛋白质组合去重
all_pair_df = (all_pair_df.dropna(subset=['INCHIKEY', 'UNIPROTID']).
               drop_duplicates(subset=['INCHIKEY', 'UNIPROTID'], keep='first')
               .reset_index(drop=True))
print('After deduplication:', len(all_pair_df))

Before deduplication: 58534
After deduplication: 49520


In [5]:
tqdm.pandas()
def cas_to_inchikey(cas):
    if pd.isna(cas) or str(cas).strip() == '':
        return pd.NA

    try:
        results = pcp.get_properties(
            'InChIKey',
            str(cas).strip(),
            namespace='name'
        )
        return results[0]['InChIKey'] if results else pd.NA
    except Exception as e:
        print(f'Failed: {cas} | {e}')
        return pd.NA

np_df = pd.read_excel('../TPP/DrugInfo.xlsx')
np_df['INCHIKEY'] = np_df['CAS'].progress_apply(cas_to_inchikey)

100%|██████████| 237/237 [01:40<00:00,  2.36it/s]


In [6]:
# 提取所有天然产物的InChIKey
np_inchikeys = set(np_df['INCHIKEY'].astype('string').str.strip().str.upper().dropna())

# 统一all_pair_df中的InChIKey格式
all_pair_df['INCHIKEY'] = (all_pair_df['INCHIKEY'].astype('string').str.strip().str.upper())

# 找到并保存被删除的药物–蛋白质对
remove_mask = all_pair_df['INCHIKEY'].isin(np_inchikeys)
removed_pair_df = all_pair_df[remove_mask].copy()

# 去除天然产物药物
all_pair_df = all_pair_df[~remove_mask].reset_index(drop=True)
all_pair_df['DRUGID'] = all_pair_df['INCHIKEY']

print(f"Removed drugs: {removed_pair_df['INCHIKEY'].nunique()}")
print(f"Removed pairs: {len(removed_pair_df)}")
print(f"Remaining pairs: {len(all_pair_df)}")

save_dir = './final_data/'
os.makedirs(save_dir, exist_ok=True)
all_pair_df.to_excel(save_dir + 'SDT.xlsx', index=False)

print('After filtering, number of proteins:', len(all_pair_df['UNIPROTID'].unique()))
print('After filtering, number of drugs:', len(all_pair_df['INCHIKEY'].unique()))
print('After filtering, number of pairs:', len(all_pair_df))

Removed drugs: 52
Removed pairs: 242
Remaining pairs: 49278
After filtering, number of proteins: 3206
After filtering, number of drugs: 24200
After filtering, number of pairs: 49278


In [7]:
# 读取数据
pair_df = pd.read_excel(r'./final_data/SDT.xlsx')
print('Before filtering, number of pairs:', len(pair_df))

# ===== 1. 构建字典 =====
drug_df = pair_df[['DRUGID', 'DRUGNAME', 'INCHIKEY', 'SMILES']].drop_duplicates()

protein_df = pair_df[['UNIPROTID', 'GENENAME', 'SEQUENCE']].drop_duplicates()

drug_dict = dict(zip(drug_df['DRUGID'], drug_df['SMILES']))
protein_dict = dict(zip(protein_df['UNIPROTID'], protein_df['SEQUENCE']))

print('Number of drugs:', len(drug_dict))
print('Number of proteins:', len(protein_dict))

# 保存字典
drug_df.to_excel(r'./final_data/DrugInfo.xlsx', index=False)
protein_df.to_excel(r'./final_data/ProtInfo.xlsx', index=False)

# ===== 2. 正样本 =====
pos_pairs = set(zip(pair_df['DRUGID'], pair_df['UNIPROTID']))
num_pos = len(pos_pairs)
print('Number of positive pairs:', num_pos)

# ===== 3. 生成全部负样本 =====
all_drugs = pair_df['DRUGID'].dropna().unique()
all_proteins = pair_df['UNIPROTID'].dropna().unique()

print('Generating all candidate negative pairs...')
all_pairs = set(product(all_drugs, all_proteins))
all_neg_pairs = list(all_pairs - pos_pairs)

print('Total candidate negative pairs:', len(all_neg_pairs))

ratios = [1, 3, 5, 7, 10]
seeds = [1, 11, 111, 1111, 11111]

for ratio in ratios:
    print(f'Generating dataset with ratio 1:{ratio}...')
    save_dir = f'./final_data/1_{ratio}'
    os.makedirs(save_dir, exist_ok=True)

    num_neg = num_pos * ratio

    for i, seed in enumerate(seeds):
        print(f'Generating dataset {i+1} with seed {seed}...')
        random.seed(seed)
        sampled_neg = random.sample(all_neg_pairs, num_neg)

        neg_df = pd.DataFrame(sampled_neg, columns=['DRUGID', 'UNIPROTID'])
        neg_df['Label'] = 0

        pos_df = pd.DataFrame(list(pos_pairs), columns=['DRUGID', 'UNIPROTID'])
        pos_df['Label'] = 1

        dataset = pd.concat([pos_df, neg_df], ignore_index=True)
        dataset = dataset.sample(frac=1, random_state=seed).reset_index(drop=True)
        dataset.to_excel(os.path.join(save_dir, f'Set{i+1}.xlsx'), index=False)

print('All datasets generated!')

Before filtering, number of pairs: 49278
Number of drugs: 24200
Number of proteins: 3206
Number of positive pairs: 49278
Generating all candidate negative pairs...
Total candidate negative pairs: 77535922
Generating dataset with ratio 1:1...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:3...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:5...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:7...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Gene